# 02. Витрины данных

Сырые таблицы неудобны для продуктового анализа: расчёт воронки по ним требует соединений и оконных функций в каждом запросе. Формируются две витрины, используемые в ноутбуках 03 и 04.

**`mart_users`** — одна строка на пользователя: временные отметки всех шагов воронки, счётчики активности и агрегаты по успешным платежам. На такой таблице конверсия любого шага рассчитывается как доля непустых значений.

**`mart_events`** — поток событий, к каждому добавлено предыдущее событие пользователя и его время. Используется для анализа переходов: определения источников входа в каталог и последовательностей, предшествующих оплате.

In [1]:
import sqlite3
from contextlib import closing

import pandas as pd

DB_PATH = '../data/skillforge.db'

In [2]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    for table in ['users', 'user_events', 'transactions']:
        display(pd.read_sql_query(f'SELECT * FROM {table} LIMIT 5;', conn))

,user_id,registration_dt,traffic_channel,device_type,country
0,10001,2026-01-01 00:00:37,paid_search,desktop,RU
1,10002,2026-01-01 00:15:17,targeted_ads,mobile,KZ
2,10003,2026-01-01 00:27:38,organic,desktop,RU
3,10004,2026-01-01 00:53:16,paid_search,desktop,BY
4,10005,2026-01-01 00:53:32,influencers,desktop,RU


,event_id,user_id,event_name,event_dt
0,1000001,10001,view_landing,2025-12-31 23:54:37
1,1000002,10001,start_registration,2025-12-31 23:59:37
2,1000003,10001,complete_registration,2026-01-01 00:00:37
3,1000004,10001,start_onboarding,2026-01-01 00:01:02
4,1000005,10001,complete_step_1,2026-01-01 00:02:56


,transaction_id,user_id,payment_dt,amount,subscription_type,payment_status
0,5000001,10002,2026-01-01 01:48:24,5900.0,one_time_course,success
1,5000002,10006,2026-01-01 02:32:03,5900.0,one_time_course,success
2,5000003,10006,2026-02-02 02:32:03,19900.0,annual_sub,success
3,5000004,10006,2026-02-28 02:32:03,1490.0,monthly_sub,success
4,5000005,10006,2026-04-04 02:32:03,1490.0,monthly_sub,success


In [3]:
USER_MART_QUERY = """
WITH user_events_agg AS (
    SELECT
        user_id,
        MIN(CASE WHEN event_name = 'start_onboarding' THEN event_dt END) AS onboarding_start_dt,
        MIN(CASE WHEN event_name = 'complete_step_1' THEN event_dt END) AS step_1_dt,
        MIN(CASE WHEN event_name = 'complete_step_2' THEN event_dt END) AS step_2_dt,
        MIN(CASE WHEN event_name = 'complete_step_3' THEN event_dt END) AS step_3_dt,
        MIN(CASE WHEN event_name = 'complete_onboarding' THEN event_dt END) AS onboarding_complete_dt,
        MIN(CASE WHEN event_name = 'view_catalog' THEN event_dt END) AS catalog_view_dt,
        MIN(CASE WHEN event_name = 'start_trial_lesson' THEN event_dt END) AS trial_start_dt,
        MIN(CASE WHEN event_name = 'finish_trial_lesson' THEN event_dt END) AS trial_finish_dt,
        MIN(CASE WHEN event_name = 'view_paywall' THEN event_dt END) AS paywall_view_dt,
        SUM(CASE WHEN event_name = 'view_catalog' THEN 1 ELSE 0 END) AS catalog_views,
        COUNT(*) AS events_count,
        COUNT(DISTINCT event_name) AS unique_event_types
    FROM user_events
    GROUP BY user_id
),

successful_payments AS (
    SELECT
        user_id,
        MIN(payment_dt) AS first_payment_dt,
        SUM(amount) AS total_revenue,
        COUNT(*) AS payments_count
    FROM transactions
    WHERE payment_status = 'success'
    GROUP BY user_id
)

SELECT
    u.user_id,
    u.registration_dt,
    u.traffic_channel,
    u.device_type,
    u.country,

    e.onboarding_start_dt,
    e.step_1_dt,
    e.step_2_dt,
    e.step_3_dt,
    e.onboarding_complete_dt,

    e.catalog_view_dt,
    e.trial_start_dt,
    e.trial_finish_dt,
    e.paywall_view_dt,

    COALESCE(e.catalog_views, 0) AS catalog_views,
    COALESCE(e.events_count, 0) AS events_count,
    COALESCE(e.unique_event_types, 0) AS unique_event_types,

    p.first_payment_dt,
    COALESCE(p.total_revenue, 0) AS total_revenue,
    COALESCE(p.payments_count, 0) AS payments_count

FROM users AS u
LEFT JOIN user_events_agg AS e ON u.user_id = e.user_id
LEFT JOIN successful_payments AS p ON u.user_id = p.user_id;
"""

with closing(sqlite3.connect(DB_PATH)) as conn:
    df_user_mart = pd.read_sql_query(USER_MART_QUERY, conn)

print(f'mart_users: {len(df_user_mart):,} строк, {df_user_mart.shape[1]} колонок')
df_user_mart.head()

mart_users: 25,000 строк, 20 колонок


,user_id,registration_dt,traffic_channel,device_type,country,onboarding_start_dt,step_1_dt,step_2_dt,step_3_dt,onboarding_complete_dt,catalog_view_dt,trial_start_dt,trial_finish_dt,paywall_view_dt,catalog_views,events_count,unique_event_types,first_payment_dt,total_revenue,payments_count
0,10001,2026-01-01 00:00:37,paid_search,desktop,RU,2026-01-01 00:01:02,2026-01-01 00:02:56,2026-01-01 00:05:45,2026-01-01 00:06:59,2026-01-01 00:07:14,2026-01-01 00:41:14,None,None,None,1,9,9,None,0.0,0
1,10002,2026-01-01 00:15:17,targeted_ads,mobile,KZ,2026-01-01 00:16:04,2026-01-01 00:16:24,None,None,None,2026-01-01 00:45:24,2026-01-01 00:48:24,2026-01-01 01:27:24,2026-01-01 01:29:24,1,9,9,2026-01-01 01:48:24,5900.0,1
2,10003,2026-01-01 00:27:38,organic,desktop,RU,2026-01-01 00:28:10,2026-01-01 00:28:35,None,None,None,None,None,None,None,0,5,5,None,0.0,0
3,10004,2026-01-01 00:53:16,paid_search,desktop,BY,2026-01-01 00:54:06,2026-01-01 00:55:12,2026-01-01 00:55:59,2026-01-01 00:56:48,2026-01-01 00:57:03,None,None,None,None,0,8,8,None,0.0,0
4,10005,2026-01-01 00:53:32,influencers,desktop,RU,2026-01-01 00:54:22,None,None,None,None,2026-01-01 01:18:22,2026-01-01 01:23:22,None,None,1,6,6,None,0.0,0


## Витрина пользователей

Конструкция `MIN(CASE WHEN event_name = ... THEN event_dt END)` преобразует вертикальный лог событий в горизонтальную таблицу шагов. Функция `MIN` выбрана не случайно: у пользователя может быть несколько просмотров каталога или пейволла, тогда как для воронки значим первый — момент достижения шага.

Платежи агрегируются отдельным подзапросом и только со статусом `success`: отказы и возвраты не должны учитываться в выручке. `LEFT JOIN` сохраняет всех пользователей, включая не совершивших ни одного события после регистрации, а `COALESCE` приводит их отсутствующие счётчики к нулю.

In [4]:
EVENT_MART_QUERY = """
SELECT
    user_id,
    event_id,
    event_name,
    event_dt,
    LAG(event_name) OVER (PARTITION BY user_id ORDER BY event_dt, event_id) AS previous_event,
    LAG(event_dt) OVER (PARTITION BY user_id ORDER BY event_dt, event_id) AS previous_event_dt
FROM user_events
ORDER BY user_id, event_dt, event_id;
"""

with closing(sqlite3.connect(DB_PATH)) as conn:
    df_event_mart = pd.read_sql_query(EVENT_MART_QUERY, conn)

print(f'mart_events: {len(df_event_mart):,} строк')
df_event_mart.head(10)

mart_events: 170,219 строк


,user_id,event_id,event_name,event_dt,previous_event,previous_event_dt
0,10001,1000001,view_landing,2025-12-31 23:54:37,None,None
1,10001,1000002,start_registration,2025-12-31 23:59:37,view_landing,2025-12-31 23:54:37
2,10001,1000003,complete_registration,2026-01-01 00:00:37,start_registration,2025-12-31 23:59:37
3,10001,1000004,start_onboarding,2026-01-01 00:01:02,complete_registration,2026-01-01 00:00:37
4,10001,1000005,complete_step_1,2026-01-01 00:02:56,start_onboarding,2026-01-01 00:01:02
5,10001,1000006,complete_step_2,2026-01-01 00:05:45,complete_step_1,2026-01-01 00:02:56
6,10001,1000007,complete_step_3,2026-01-01 00:06:59,complete_step_2,2026-01-01 00:05:45
7,10001,1000008,complete_onboarding,2026-01-01 00:07:14,complete_step_3,2026-01-01 00:06:59
8,10001,1000009,view_catalog,2026-01-01 00:41:14,complete_onboarding,2026-01-01 00:07:14
9,10002,1000010,view_landing,2025-12-31 23:55:17,None,None


In [5]:
with closing(sqlite3.connect(DB_PATH)) as conn:
    df_user_mart.to_sql('mart_users', conn, if_exists='replace', index=False)
    df_event_mart.to_sql('mart_events', conn, if_exists='replace', index=False)
    conn.commit()

    saved = pd.read_sql_query('''
        SELECT 'mart_users' AS mart, COUNT(*) AS rows FROM mart_users
        UNION ALL SELECT 'mart_events', COUNT(*) FROM mart_events;
    ''', conn)

saved

,mart,rows
0,mart_users,25000
1,mart_events,170219


## Витрина событий

Функция `LAG` с сортировкой по `event_dt, event_id` возвращает предыдущее событие в рамках пользователя. Сортировка по двум полям, а не только по времени, обусловлена точностью отметок: события фиксируются с точностью до секунды, и при совпадении времени порядок должен определяться идентификатором, иначе состав переходов будет меняться от запуска к запуску.

## Сохранение витрин

Витрины записываются в ту же базу с параметром `if_exists='replace'`: пересчёт витрин не должен зависеть от того, выполнялся ли ноутбук ранее.